# Which settings matter? (#25)

Observational analysis of the run history: hot full cycles only, joined with the
vision scores. All logic lives in `maus_export.analysis` (tested); this notebook
only tells the story and draws the plots.

**Read with care:** settings were tuned by hand, often together, so this shows
*associations* and which settings can't be told apart. The designed experiment
(#26) is what confirms them.

Outputs must be cleared before committing (public repo):
`jupyter nbconvert --clear-output --inplace notebooks/*.ipynb`

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from maus_export import analysis as an

DATA = an.find_data_dir()  # $MAUS_DATA_DIR, ../data, or the main checkout
N_BOOT = int(os.environ.get("MAUS_N_BOOT", an.N_BOOT))
GRID, INK2 = "#e1e0d9", "#52514e"
GOOD, BAD = "#2a78d6", "#eb6834"

runs = pd.read_parquet(DATA / "runs.parquet")
vision_file = DATA / "vision.parquet"
vision = (
    pd.read_parquet(vision_file) if vision_file.is_file() else pd.DataFrame()
)
frame = an.analysis_frame(runs, vision)
factors = an.factor_columns(frame)
print(f"{len(frame)} hot full cycles, {len(factors)} varying settings")

## 1. Outcomes
Success rate per robot and topping (full cycles with a hot plate).

In [ ]:
rates = frame.groupby(["robot", "topping"])["success"].agg(["mean", "size"])
rates.columns = ["success_rate", "runs"]
display(rates.round(2))

fig, ax = plt.subplots(figsize=(7, 3))
by_robot = frame.groupby("robot")["success"].mean().sort_values()
ax.barh(by_robot.index, by_robot.values, color=GOOD, height=0.5)
ax.set_xlim(0, 1)
ax.set_xlabel("share of full cycles that succeeded")
ax.grid(axis="x", color=GRID)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.show()

## 2. What moved together
Settings correlated at |r| >= 0.8 across runs form one group: the data cannot say
which member matters, so effects are reported per group.

In [ ]:
corr, groups = an.confounding(frame, factors)
names = [an.short(f).split(".")[-1] for f in corr.columns]
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(names)), names, rotation=90, fontsize=8)
ax.set_yticks(range(len(names)), names, fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.7, label="correlation across runs")
plt.show()
for g in groups:
    if len(g) > 1:
        print("moved together:", an.group_name(g))

## 3. One-setting changes (natural experiments)
Consecutive settings blocks on one robot where only one setting (or group)
changed: last runs before vs first runs after. Few exist because most tuning
steps lasted only 1-2 runs.

In [ ]:
one = an.one_change_effects(frame, groups)
cols = ["robot", "factor", "from", "to", "n_before", "n_after"]
cols += [
    c
    for c in one.columns
    if c.startswith(("success.", "spread_ok.", "fold1_ok."))
]
display(one[cols].round(3) if len(one) else "no one-setting changes")

## 4. Effects across all runs
Regularised model per outcome (logistic for pass/fail, ridge for scores) with
robot, topping, hotplate and room temperature as covariates. Bars: 90% bootstrap
interval over whole segments; dots: effect of +1 standard deviation of the
setting. Filled dots: interval excludes zero.

In [ ]:
eff = an.model_effects(frame, groups, n_boot=N_BOOT)
show = [
    "success",
    "spread_ok",
    "fold1_ok",
    "fold2_ok",
    "spread_diameter",
    "fold2_area_ratio",
]
show = [o for o in show if o in set(eff["outcome"])]
fig, axes = plt.subplots(
    1, len(show), figsize=(3.2 * len(show), 6), sharey=True
)
order = sorted(eff["group"].unique())
label = {
    g: (g if len(g) < 40 else g[:37] + "...").replace("dispense.batter.", "")
    for g in order
}
for ax, outcome in zip(np.atleast_1d(axes), show, strict=True):
    e = eff[eff["outcome"] == outcome].set_index("group").reindex(order)
    y = np.arange(len(order))
    clear = (e["lo"] > 0) | (e["hi"] < 0)
    ax.hlines(y, e["lo"], e["hi"], color=INK2, lw=1.5)
    ax.scatter(
        e["coef_per_sd"],
        y,
        s=30,
        zorder=3,
        color=np.where(clear, GOOD, "white"),
        edgecolors=GOOD,
    )
    ax.axvline(0, color=GRID, lw=1)
    better = an.OUTCOMES[outcome].better
    ax.set_title(
        f"{outcome}\n({'higher' if better > 0 else 'lower'} is better)",
        fontsize=10,
    )
    ax.grid(axis="x", color=GRID)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
np.atleast_1d(axes)[0].set_yticks(
    range(len(order)), [label[g] for g in order], fontsize=8
)
plt.tight_layout()
plt.show()

## 5. Shortlist for the designed experiment (#26)
Settings with a consistent direction, ranked by evidence. The safe range is the
span of values that produced successful crepes, so #26 never proposes untested
values. `explore = False` means only one value ever succeeded.

In [ ]:
top = an.shortlist(frame, groups, eff, one)
table = pd.DataFrame(
    [
        {
            "settings": t["group"].replace("dispense.batter.", ""),
            "direction": t["direction"],
            "helps": ", ".join(t["helps"]),
            "hurts": ", ".join(t["hurts"]),
            "one-change confirmed": bool(t["one_change_support"]),
            "explore": t["explore"],
            "safe range": "; ".join(
                f"{k.split('.')[-1]} {v[0]:g}..{v[1]:g}"
                for k, v in t["safe_range"].items()
            ),
            "current": "; ".join(
                f"{k.split('.')[-1]} {v:g}" for k, v in t["current"].items()
            ),
        }
        for t in top
    ]
)
display(table)

## What this means
- Each shortlisted group is a candidate factor for #26, with the safe range as
  its level bounds.
- Groups that moved together must be varied **independently** in #26, otherwise
  the experiment repeats the history's blind spot.
- Trade-offs (raising helps one outcome, lowering another) need the experiment
  to find the balance point.